# 📦 AI-Based Smart Product Identification System
#### Supervisor: Dr. Zaki Hassan | Group 13

### Project Objective
Traditional computer vision models only classify the entire room image or make single-label predictions. In contrast, this project implements a **two-stage perception and reasoning architecture**:
1. **Fine-Grained Object Detection & Spatial Counting**: Automatically locates, counts, and tracks every distinct furniture object (e.g., sofas, chairs, beds, dining tables, electronics, appliances).
2. **Semantic Scene & Room Prediction**: Infers the room function (Living Room, Bedroom, Dining Room, Study, Kitchen) based on empirical furniture distributions, counts, and estimated seating capacities.

In [ ]:
# 1. Install & import necessary computer vision libraries
!pip install opencv-python onnxruntime pillow numpy matplotlib -q

import cv2
import numpy as np
import onnxruntime as ort
import matplotlib.pyplot as plt
from PIL import Image
import urllib.request
import os

print("All libraries successfully loaded!")

## 2. Load the High-Performance YOLO Deep Learning Model
We load the latest ONNX-optimized YOLO model for instant bounding box prediction and object categorization.

In [ ]:
model_path = 'yolo11n.onnx'
if not os.path.exists(model_path):
    url = 'https://github.com/ultralytics/assets/releases/download/v8.4.0/yolo11n.onnx'
    print("Downloading model checkpoint...")
    urllib.request.urlretrieve(url, model_path)

session = ort.InferenceSession(model_path)
input_name = session.get_inputs()[0].name
output_name = session.get_outputs()[0].name
print(f"Model loaded. Ready for inference!")

## 3. Core Preprocessing, Inference & Object Counting Pipeline

In [ ]:
COCO_CLASSES = [
    'person', 'bicycle', 'car', 'motorcycle', 'airplane', 'bus', 'train', 'truck', 'boat',
    'traffic light', 'fire hydrant', 'stop sign', 'parking meter', 'bench', 'bird', 'cat',
    'dog', 'horse', 'sheep', 'cow', 'elephant', 'bear', 'zebra', 'giraffe', 'backpack',
    'umbrella', 'handbag', 'tie', 'suitcase', 'frisbee', 'skis', 'snowboard', 'sports ball',
    'kite', 'baseball bat', 'baseball glove', 'skateboard', 'surfboard', 'tennis racket',
    'bottle', 'wine glass', 'cup', 'fork', 'knife', 'spoon', 'bowl', 'banana', 'apple',
    'sandwich', 'orange', 'broccoli', 'carrot', 'hot dog', 'pizza', 'donut', 'cake',
    'chair', 'couch', 'potted plant', 'bed', 'dining table', 'toilet', 'tv', 'laptop',
    'mouse', 'remote', 'keyboard', 'cell phone', 'microwave', 'oven', 'toaster', 'sink',
    'refrigerator', 'book', 'clock', 'vase', 'scissors', 'teddy bear', 'hair drier', 'toothbrush'
]

OBJECT_MAP = {
    'chair': 'Chair',
    'couch': 'Sofa / Couch',
    'bed': 'Bed',
    'dining table': 'Dining / Coffee Table',
    'tv': 'Television',
    'potted plant': 'Indoor Plant',
    'refrigerator': 'Refrigerator',
    'microwave': 'Microwave',
    'oven': 'Oven / Stove',
    'sink': 'Sink',
    'toilet': 'Toilet',
    'laptop': 'Laptop'
}

def analyze_room_image(image_path, conf_thresh=0.25):
    img_bgr = cv2.imread(image_path)
    h, w = img_bgr.shape[:2]
    
    # Preprocess
    scale = min(640 / h, 640 / w)
    new_w, new_h = int(w * scale), int(h * scale)
    resized = cv2.resize(img_bgr, (new_w, new_h))
    canvas = np.full((640, 640, 3), 114, dtype=np.uint8)
    dx, dy = (640 - new_w) // 2, (640 - new_h) // 2
    canvas[dy:dy+new_h, dx:dx+new_w] = resized
    
    blob = cv2.cvtColor(canvas, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
    blob = np.transpose(blob, (2, 0, 1))[None, ...]
    
    # Inference
    outputs = session.run([output_name], {input_name: blob})[0]
    preds = outputs[0].transpose(1, 0)
    
    boxes, confidences, class_ids = [], [], []
    for row in preds:
        scores = row[4:]
        cls = int(np.argmax(scores))
        c = float(scores[cls])
        if c >= conf_thresh:
            cx, cy, bw, bh = row[:4]
            x1 = (cx - bw/2 - dx) / scale
            y1 = (cy - bh/2 - dy) / scale
            boxes.append([int(x1), int(y1), int(bw/scale), int(bh/scale)])
            confidences.append(c)
            class_ids.append(cls)
            
    indices = cv2.dnn.NMSBoxes(boxes, confidences, conf_thresh, 0.45)
    
    counts = {}
    annotated = img_bgr.copy()
    if len(indices) > 0:
        for idx in indices.flatten():
            label = COCO_CLASSES[class_ids[idx]]
            if label in OBJECT_MAP:
                counts[label] = counts.get(label, 0) + 1
                x, y, bw, bh = boxes[idx]
                cv2.rectangle(annotated, (x, y), (x + bw, y + bh), (0, 255, 0), 3)
                cv2.putText(annotated, f"{OBJECT_MAP[label]} {confidences[idx]:.2f}", (x, max(25, y-8)),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
                
    # Predict Room Type
    scores = {
        'Bedroom': counts.get('bed', 0) * 18 + counts.get('tv', 0) * 3,
        'Living Room': counts.get('couch', 0) * 15 + counts.get('tv', 0) * 8 + counts.get('dining table', 0) * 3 + counts.get('chair', 0) * 2,
        'Dining Room': counts.get('dining table', 0) * 15 + counts.get('chair', 0) * 6,
        'Kitchen': counts.get('refrigerator', 0) * 20 + counts.get('oven', 0) * 15 + counts.get('sink', 0) * 10,
        'Bathroom': counts.get('toilet', 0) * 25 + counts.get('sink', 0) * 12
    }
    best_room = max(scores, key=scores.get) if sum(scores.values()) > 0 else 'General Hall / Room'
    
    return counts, best_room, cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB)

## 4. Run Analysis & Display Supervisor-Ready Results

In [ ]:
test_image = 'sample_interior.jpg'
counts, room_type, annotated_img = analyze_room_image(test_image)

print("=" * 60)
print(f"🎯 PREDICTED ROOM: {room_type}")
print(f"📊 DETECTED OBJECT COUNTS:")
for k, v in counts.items():
    print(f"   - {OBJECT_MAP.get(k, k)}: {v}")
print("=" * 60)

plt.figure(figsize=(10, 7))
plt.imshow(annotated_img)
plt.axis('off')
plt.title(f"Room Type: {room_type} | Counts: {counts}")
plt.show()